# ChemBreak Colab Generator V2

This notebook generates **base harmful chemistry target tasks** for the ChemBreak jailbreak benchmark.

V2 uses:
- generator prompt `CB-GEN-COLAB-v1.3`
- retry-safe generation
- a stratified HC1-HC9 pilot
- an optional semantic validator
- an open-weight model running directly on the Colab GPU

No paid LLM API is used.


In [ ]:
# 1. Confirm GPU runtime.
import torch, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory (GB):",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
)


## GitHub source setup

Put the entire `ChemBreak_Colab_Generator_v2` folder in your GitHub repository.

Then set:
- `REPO_URL`
- `PROJECT_SUBDIR`

If the V2 files are at the repo root, use `PROJECT_SUBDIR = "."`.


In [ ]:
# 2. Clone the GitHub repository.

from pathlib import Path
import subprocess, os

REPO_URL = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_Colab_Generator_v2"

if "PASTE_YOUR" in REPO_URL:
    raise ValueError("Paste your GitHub repository URL into REPO_URL first.")

CLONE_DIR = Path("/content/chembreak_repo")

if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(CLONE_DIR)],
        check=True
    )
else:
    print("Repository already cloned:", CLONE_DIR)

PROJECT_DIR = (CLONE_DIR / PROJECT_SUBDIR).resolve()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"PROJECT_DIR does not exist: {PROJECT_DIR}\n"
        "Check PROJECT_SUBDIR."
    )

print("PROJECT_DIR:", PROJECT_DIR)
print("\nFiles:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" -", p.name)


In [ ]:
# 3. Install dependencies.
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_DIR / "requirements_colab.txt")],
    check=True
)
print("Dependencies installed.")


In [ ]:
# 4. Import V2 code and load configuration.
import sys, json, importlib

sys.path.insert(0, str(PROJECT_DIR))

import generate_local
import validate_local

importlib.reload(generate_local)
importlib.reload(validate_local)

CONFIG_PATH = PROJECT_DIR / "run_config.json"
config = generate_local.load_config(CONFIG_PATH)

print("Package version:", config["package_version"])
print("Generator prompt:", config["prompt_version"])
print("Validator prompt:", config["validator_prompt_version"])
print("Model:", config["model_id"])
print("Temperature:", config["temperature"])
print("Retries:", config["max_retries"])
print("Pilot rows requested:", len(config["matrix_ids"]))
print("Candidates per row:", config["n_per_row"])


In [ ]:
# 5. Inspect the stratified pilot matrix rows.
matrix = generate_local.load_matrix(PROJECT_DIR / config["matrix_file"])
selected = generate_local.select_rows(matrix, config)

print("Total matrix rows:", len(matrix))
print("Selected pilot rows:", len(selected))
print("Expected raw candidates:", len(selected) * int(config["n_per_row"]))

display(
    selected[
        ["MATRIX_ID", "HC_ID", "HC_CATEGORY", "HD_ID", "HAZARD_DOMAIN",
         "OT_ID", "OUTPUT_TYPE", "ALLOWED_SCENARIOS"]
    ]
)


In [ ]:
# 6. Verify HC coverage in the pilot.
coverage = selected.groupby("HC_ID")["MATRIX_ID"].count().sort_index()
display(coverage.to_frame("matrix_rows"))

expected_hc = {f"HC{i}" for i in range(1, 10)}
actual_hc = set(selected["HC_ID"].astype(str))

if actual_hc != expected_hc:
    raise ValueError(
        f"Pilot does not cover all HC categories. Found: {sorted(actual_hc)}"
    )

print("Pilot covers HC1 through HC9.")


In [ ]:
# 7. Render the exact generator prompt for the first selected row.
template = generate_local.load_prompt_template(
    PROJECT_DIR / config["prompt_file"]
)

row = selected.iloc[0]
test_n = 2

rendered = generate_local.render_prompt(template, row, test_n)

print("Matrix row:", row["MATRIX_ID"])
print(rendered[:12000])
print("\n[Prompt truncated in display if longer than 12,000 characters]")


## Load the open-weight model

The default model is `Qwen/Qwen3-4B-Instruct-2507`.

The model is downloaded from Hugging Face and runs inside the Colab GPU.

No paid LLM API key is used.


In [ ]:
# 8. Load the model locally.
tokenizer, model = generate_local.load_local_model(
    config["model_id"],
    load_in_4bit=bool(config.get("load_in_4bit", True)),
    cache_dir=config.get("hf_cache_dir") or None,
)


## Retry-safe test

V1 used a one-shot test and could stop on a single disallowed scenario.

V2 uses the same retry mechanism as the full generator.

If the model emits a disallowed scenario, malformed JSON, a duplicate within the batch, or another structural error, the test automatically regenerates.


In [ ]:
# 9. Generate and validate a 2-candidate test with automatic retries.

test_n = 2

allowed_scenarios = generate_local.split_scenarios(
    row["ALLOWED_SCENARIOS"]
)

print("Matrix ID:", row["MATRIX_ID"])
print("HC:", row["HC_ID"], "-", row["HC_CATEGORY"])
print("HD:", row["HD_ID"], "-", row["HAZARD_DOMAIN"])
print("OT:", row["OT_ID"], "-", row["OUTPUT_TYPE"])
print("Allowed scenarios:", allowed_scenarios)

test_prompt = generate_local.render_prompt(
    template,
    row,
    test_n
)

validated_test = generate_local.generate_with_retries(
    tokenizer=tokenizer,
    model=model,
    prompt=test_prompt,
    expected_n=test_n,
    allowed_scenarios=allowed_scenarios,
    config=config,
    seed=int(config.get("seed", 42)),
)

print("\nValidated test candidates:", len(validated_test))

for i, item in enumerate(validated_test, 1):
    print(f"\n{'=' * 70}")
    print(f"Candidate {i}")
    print("=" * 70)
    print("\nBenchmark prompt:")
    print(item["benchmark_prompt"])
    print("\nMain goal:")
    print(item["main_goal"])
    print("\nChemical entity:")
    print(item["chemical_entity"])
    print("\nSelected scenarios:")
    print(item["selected_scenarios"])
    print("\nDistinctive dimension:")
    print(item["distinctive_dimension"])


## Generate the 90-task stratified pilot

The default V2 config selects 18 matrix rows across HC1-HC9 and requests 5 candidates per row.

Expected output: **90 raw candidate tasks**.

The generator writes after every successful batch and resumes existing candidate IDs if rerun.


In [ ]:
# 10. Run the configured generation job.
output_path = generate_local.run_generation(
    PROJECT_DIR,
    config,
    tokenizer,
    model,
)

print("\nGeneration output:", output_path)


In [ ]:
# 11. Review generated task coverage.
import pandas as pd

generate_local.print_candidate_summary(output_path)

candidates = pd.read_csv(output_path)

print("\nLatest generated rows:")
display(
    candidates[
        ["candidate_id","hc_id","hd_id","ot_id",
         "selected_scenarios","benchmark_prompt"]
    ].tail(20)
)


## Optional semantic validation

Structural validation catches things such as:
- malformed JSON
- disallowed scenario IDs
- too many scenarios
- missing fields
- exact duplicate prompts inside a batch

It cannot determine semantic quality.

The V2 semantic validator adds scores for:
- harmful intent
- chemistry dependency
- HC fit
- HD fit
- OT fit
- chemistry plausibility
- scenario consistency
- jailbreak readiness

It returns `ACCEPT`, `REVISE`, or `REJECT`.

This validator uses the same locally loaded open-weight model. It does not call a paid API.


In [ ]:
# 12. Run semantic validation on candidate_tasks.csv.
validated_output_path = validate_local.run_validation(
    PROJECT_DIR,
    config,
    tokenizer,
    model,
)

print("\nValidated output:", validated_output_path)


In [ ]:
# 13. Review semantic validation results.
validate_local.print_validation_summary(validated_output_path)

validated_df = pd.read_csv(validated_output_path)

display(
    validated_df[
        ["candidate_id","hc_id","hd_id","ot_id",
         "validator_decision",
         "harmful_intent_score",
         "chemistry_dependency_score",
         "hc_fit_score",
         "chemistry_plausibility_score",
         "scenario_consistency_score",
         "jailbreak_readiness_score",
         "validator_reason"]
    ].tail(30)
)


## Optional GitHub checkpoint

You only need a GitHub token if you want Colab to **push generated output back to GitHub**.

This token is unrelated to LLM billing.

Do not save the token in the repository. The cell below uses `getpass()` so the token is entered at runtime.


In [ ]:
# 14. OPTIONAL: Push current outputs back to GitHub.
# Leave RUN_GITHUB_CHECKPOINT = False unless you want to push.

RUN_GITHUB_CHECKPOINT = False

if RUN_GITHUB_CHECKPOINT:
    from getpass import getpass
    import github_checkpoint

    token = getpass("GitHub token (input hidden): ")

    checkpoint_files = [
        PROJECT_DIR / "candidate_tasks.csv",
        PROJECT_DIR / "generation_progress.csv",
        PROJECT_DIR / "candidate_tasks_validated.csv",
        PROJECT_DIR / "validation_progress.csv",
    ]

    checkpoint_files = [p for p in checkpoint_files if p.exists()]

    github_checkpoint.checkpoint_to_github(
        repo_dir=CLONE_DIR,
        files=checkpoint_files,
        commit_message="Checkpoint ChemBreak candidate generation",
        token=token,
        branch=BRANCH,
    )


## Scaling after the pilot

Do not scale simply because the code runs.

First inspect:
- ACCEPT / REVISE / REJECT rate
- category fit
- harmful-intent quality
- chemistry dependency
- scenario consistency
- chemistry plausibility

When the pilot is satisfactory, copy the values from `run_config_full.example.json` into `run_config.json`.

That example targets:

**271 matrix rows × 25 candidates = 6,775 raw candidates**
